# 01 — Data Preparation and Dataset Splits
Run this notebook first. It reads the five EPA ozone archives, applies a consistent measurement definition, selects one instrument per station from the initial training period, and writes a daily panel.

**Target:** next-day daily maximum 8-hour ozone, in ppb (`1st Max Value` for the specified 8-hour series).

**Time split:** 2021–2022 initial training; 2023–2024 monthly development validation; 2025 final evaluation.
**Station split:** the frozen station manifest is an input, not re-randomized on each run. Held-out stations never enter model fitting or tuning.

Preparing 2025 data here does not use it for tuning. The modeling notebook reads only the development feature file. This is a reproduction of an existing analysis; 2025 outcomes have previously been examined.

## 1. Set up paths and libraries

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Locate code/ whether Jupyter starts in the project, code/, or code/script/.
CODE = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "code"]
             if (p / "script/01_data_preparation.ipynb").exists()), None)
if CODE is None:
    raise FileNotFoundError("Open this notebook from the project directory.")
DATA = CODE / "data"
RAW = DATA / "raw"
CONFIG = DATA / "config"
PROCESSED = DATA / "processed"
RESULT = CODE / "result"
TABLES = RESULT / "tables"
FIGURES = RESULT / "figures"
MODELS = RESULT / "models"
REPORTS = RESULT / "reports"
for folder in [PROCESSED, TABLES, FIGURES, MODELS, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Project directory:", CODE)

Project directory: /Users/yigefeng/Desktop/WashU大三上/SDS4425/Project/code


## 2. Read and filter the original archives
Use the same sample duration, pollutant standard, event type and unit throughout. Station IDs retain leading zeros. Values are converted from ppm to ppb. Mexico and Puerto Rico are excluded, matching the original station cohort.

Do not average records across different standards or instruments. Unexpected duplicate station-date-instrument records stop execution for inspection.

In [2]:
import zipfile
import hashlib
columns = ["State Code", "County Code", "Site Num", "POC", "Date Local",
           "Event Type", "Sample Duration", "Pollutant Standard",
           "Units of Measure", "1st Max Value", "State Name", "Latitude", "Longitude"]
frames, inventory = [], []
for year in range(2021, 2026):
    archive = RAW / f"daily_44201_{year}.zip"
    with zipfile.ZipFile(archive) as z:
        with z.open(f"daily_44201_{year}.csv") as handle:
            raw = pd.read_csv(handle, dtype=str, keep_default_na=False, usecols=columns)
    raw["station_id"] = (raw["State Code"].str.zfill(2) + "-" +
                         raw["County Code"].str.zfill(3) + "-" + raw["Site Num"].str.zfill(4))
    keep = (raw["Event Type"].eq("None") &
            raw["Sample Duration"].eq("8-HR RUN AVG BEGIN HOUR") &
            raw["Pollutant Standard"].eq("Ozone 8-hour 2015") &
            raw["Units of Measure"].eq("Parts per million") &
            ~raw["State Name"].isin(["Country Of Mexico", "Puerto Rico"]))
    daily = raw.loc[keep].copy()
    daily["date"] = pd.to_datetime(daily["Date Local"])
    daily["y"] = pd.to_numeric(daily["1st Max Value"], errors="coerce") * 1000
    assert not daily.duplicated(["station_id", "date", "POC"]).any()
    inventory.append({"year": year, "raw_rows": len(raw), "filtered_rows": len(daily),
                      "first_date": str(daily.date.min().date()), "last_date": str(daily.date.max().date()),
                      "sha256": hashlib.sha256(archive.read_bytes()).hexdigest()})
    frames.append(daily[["station_id", "POC", "date", "y", "State Name", "Latitude", "Longitude"]])
    print("Loaded", year)
raw_panel = pd.concat(frames, ignore_index=True)
inventory = pd.DataFrame(inventory)
inventory.to_csv(TABLES / "raw_data_inventory.csv", index=False)
display(inventory.drop(columns="sha256"))
del raw, daily, frames

Loaded 2021


Loaded 2022


Loaded 2023


Loaded 2024


Loaded 2025


,year,raw_rows,filtered_rows,first_date,last_date
0,2021,394384,381400,2021-01-01,2021-12-31
1,2022,387156,375030,2022-01-01,2022-12-31
2,2023,388607,374123,2023-01-01,2023-12-31
3,2024,386972,374604,2024-01-01,2024-12-31
4,2025,381023,372402,2025-01-01,2025-12-31


## 3. Select instruments using 2021–2022 only
Choose the instrument with the most nonmissing initial-period observations at each station. Break ties by the smallest numeric POC. Require at least 90 observed days in that period.

The reference instrument manifest is retained only as a reproducibility check; selection is recomputed from the raw training-period observations.

In [3]:
initial = raw_panel[(raw_panel.date < "2023-01-01") & raw_panel.y.notna()]
counts = initial.groupby(["station_id", "POC"]).size().rename("initial_valid_days").reset_index()
counts["poc_sort"] = pd.to_numeric(counts.POC)
instruments = counts.sort_values(["station_id", "initial_valid_days", "poc_sort"],
                                 ascending=[True, False, True]).drop_duplicates("station_id")
reference = pd.read_csv(CONFIG / "instruments_reference.csv", dtype={"station_id": str, "POC": str})
actual = instruments.set_index("station_id").POC.sort_index()
expected = reference.set_index("station_id").POC.sort_index()
assert actual.equals(expected), "Instrument choices differ from the frozen analysis."
instruments.to_csv(PROCESSED / "selected_instruments.csv", index=False)
roles = pd.read_csv(CONFIG / "station_split.csv", dtype=str)
assert roles.station_id.is_unique
assert set(roles.role) == {"development", "held_out"}
panel = raw_panel.merge(instruments[["station_id", "POC"]], on=["station_id", "POC"], validate="many_to_one")
panel = panel.merge(roles, on="station_id", validate="many_to_one")
eligible = set(instruments.loc[instruments.initial_valid_days >= 90, "station_id"])
panel = panel[panel.station_id.isin(eligible)].sort_values(["date", "station_id"]).reset_index(drop=True)
assert not panel.duplicated(["station_id", "date"]).any()
assert panel.date.min() >= pd.Timestamp("2021-01-01")
assert panel.date.max() <= pd.Timestamp("2025-12-31")
assert not set(panel.loc[panel.role == "development", "station_id"]) & set(panel.loc[panel.role == "held_out", "station_id"])
print("Eligible stations:", panel.station_id.nunique())
del raw_panel, initial

Eligible stations: 1215


## 4. Label and save the time and station splits
The development period is not a single training block: Notebook 03 constructs expanding training windows within it. A held-out station remains held out in every year.

Missing ozone values are retained in the calendar panel. Notebook 02 explicitly reindexes dates and requires complete historical windows; missing calendar days must not silently become adjacent days.

In [4]:
panel["period"] = np.select([panel.date < "2023-01-01", panel.date < "2025-01-01"],
                            ["initial_training", "development_validation"], default="final_test")
panel.to_csv(PROCESSED / "daily_panel.csv.gz", index=False, compression="gzip")
coverage = panel.groupby(["role", "period"]).agg(rows=("y", "size"),
                  observed_targets=("y", "count"), stations=("station_id", "nunique")).reset_index()
coverage.to_csv(TABLES / "data_split_summary.csv", index=False)
metadata = {"target": "daily maximum 8-hour ozone (ppb)", "initial_train_end": "2022-12-31",
            "development_end": "2024-12-31", "test_start": "2025-01-01",
            "min_initial_observations": 90, "station_manifest": "data/config/station_split.csv",
            "test_previously_examined": True, "missing_values_imputed": False}
(PROCESSED / "data_protocol.json").write_text(json.dumps(metadata, indent=2))
display(coverage)
print("Next: run 02_time_series_features.ipynb")

,role,period,rows,observed_targets,stations
0,development,development_validation,566905,566905,934
1,development,final_test,275717,275717,895
2,development,initial_training,587505,587505,971
3,held_out,development_validation,145255,145255,235
4,held_out,final_test,70267,70267,221
5,held_out,initial_training,150209,150209,244


Next: run 02_time_series_features.ipynb
